<a href="https://colab.research.google.com/github/kalvaamrutha/ML_Experiments/blob/main/EXP_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from google.colab import files

# Trigger file upload
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
display(df.head())

### 2. Preprocess Data: Target Encoding, Train-Test Split, Imputation & Scaling

In [ ]:
# Separate features and target
target = "PlacementStatus"
X = df.drop(columns=[target])
y = df[target]

# One-hot encoding of categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Replace potential infinities with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)

# Imputation
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)

print("Training set shape:", X_train_scaled.shape)
print("Validation set shape:", X_val_scaled.shape)

### 3. Compare Regularization Models

In [ ]:
def compare_regularisation(X_train, y_train, X_val, y_val, C_values):
    results = []
    for C in C_values:
        # LASSO - L1
        lasso = LogisticRegression(
            penalty="l1",
            C=C,
            solver="liblinear",
            max_iter=5000,
            random_state=42
        )
        lasso.fit(X_train, y_train)
        lasso_pred = lasso.predict(X_val)
        lasso_acc = accuracy_score(y_val, lasso_pred)
        lasso_nonzero = np.sum(np.abs(lasso.coef_) > 1e-6)

        # RIDGE - L2
        ridge = LogisticRegression(
            penalty="l2",
            C=C,
            solver="liblinear",
            max_iter=5000,
            random_state=42
        )
        ridge.fit(X_train, y_train)
        ridge_pred = ridge.predict(X_val)
        ridge_acc = accuracy_score(y_val, ridge_pred)
        ridge_nonzero = np.sum(np.abs(ridge.coef_) > 1e-6)

        # ELASTIC NET - L1 + L2
        elastic = LogisticRegression(
            penalty="elasticnet",
            C=C,
            l1_ratio=0.5,
            solver="saga",
            max_iter=5000,
            random_state=42
        )
        elastic.fit(X_train, y_train)
        elastic_pred = elastic.predict(X_val)
        elastic_acc = accuracy_score(y_val, elastic_pred)

        results.append({
            "C": C,
            "Lasso_val_acc": lasso_acc,
            "Ridge_val_acc": ridge_acc,
            "ElasticNet_val_acc": elastic_acc,
            "Lasso_nonzero": lasso_nonzero,
            "Ridge_nonzero": ridge_nonzero
        })

    return pd.DataFrame(results)

C_values = [0.001, 0.01, 0.1, 1, 10, 100]
results_df = compare_regularisation(X_train_scaled, y_train, X_val_scaled, y_val, C_values)
display(results_df)

### 4. Find Best Hyperparameters

In [ ]:
best_lasso = results_df.loc[results_df["Lasso_val_acc"].idxmax()]
best_ridge = results_df.loc[results_df["Ridge_val_acc"].idxmax()]
best_elastic = results_df.loc[results_df["ElasticNet_val_acc"].idxmax()]

print("--- Best Lasso Model ---")
print(best_lasso)
print("\n--- Best Ridge Model ---")
print(best_ridge)
print("\n--- Best Elastic Net Model ---")
print(best_elastic)

### 5. Plot Accuracy vs Regularization Strength ($C$)

In [ ]:
plt.figure(figsize=(8, 5))
plt.semilogx(results_df["C"], results_df["Lasso_val_acc"], marker="o", label="Lasso")
plt.semilogx(results_df["C"], results_df["Ridge_val_acc"], marker="o", label="Ridge")
plt.semilogx(results_df["C"], results_df["ElasticNet_val_acc"], marker="o", label="Elastic Net")

plt.xlabel("C (Inverse of Regularization Strength)")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy vs C")
plt.legend()
plt.grid(True)
plt.show()

### 6. Shrinkage Paths for Lasso and Ridge

In [ ]:
lasso_coefficients = []
ridge_coefficients = []

for C in C_values:
    # Fit Lasso
    lasso_model = LogisticRegression(penalty="l1", C=C, solver="liblinear", max_iter=5000, random_state=42)
    lasso_model.fit(X_train_scaled, y_train)
    lasso_coefficients.append(lasso_model.coef_[0])

    # Fit Ridge
    ridge_model = LogisticRegression(penalty="l2", C=C, solver="liblinear", max_iter=5000, random_state=42)
    ridge_model.fit(X_train_scaled, y_train)
    ridge_coefficients.append(ridge_model.coef_[0])

lasso_coefficients = np.array(lasso_coefficients)
ridge_coefficients = np.array(ridge_coefficients)

# Plot Lasso Shrinkage Paths
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
for i in range(lasso_coefficients.shape[1]):
    plt.semilogx(C_values, lasso_coefficients[:, i])
plt.xlabel("C")
plt.ylabel("Coefficient Value")
plt.title("Lasso Shrinkage Paths")
plt.grid(True)

# Plot Ridge Shrinkage Paths
plt.subplot(1, 2, 2)
for i in range(ridge_coefficients.shape[1]):
    plt.semilogx(C_values, ridge_coefficients[:, i])
plt.xlabel("C")
plt.ylabel("Coefficient Value")
plt.title("Ridge Shrinkage Paths")
plt.grid(True)

plt.tight_layout()
plt.show()